# 02 — Agent as Tool

This notebook demonstrates the **agent-as-tool** pattern: wrapping specialized agents as tools that an orchestrator agent can call on demand.

In example 01 we saw how `call_model()` manages conversation history across turns. Here we show the opposite: **stateless** agent invocations. `as_tool()` creates a fresh `GlobalState` for each call, so the specialized agents never see prior history — they receive a query, do focused work, and return structured output.

**Prerequisites:**
- A `.env` file with `LLM_API_KEY` and `LLM_MODEL` set (see example 01)

## 1. Setup

In [ ]:
from orqest import load_config

config = load_config()
print(f"Using model: {config.llm_model}")

Using model: openai:gpt-4.1


## 2. Define specialized agents

We create two focused agents, each with a specific job and structured output.

Note that `_run_implementation()` uses `self.call_model()` — the same method used for multi-turn in example 01. The difference is that `as_tool()` creates a **fresh `GlobalState`** per call, so `state.message_history` is always empty. The agents are stateless by how they're invoked, not by how they're written.

In [ ]:
from pydantic import BaseModel, Field
from orqest.agents import BaseAgent, GlobalState


# --- Summarization Agent ---

class SummaryOutput(BaseModel):
    summary: str = Field(description="A concise summary")
    key_points: list[str] = Field(description="Main takeaways")


class SummaryAgent(BaseAgent[GlobalState, SummaryOutput]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="summarizer",
            system_prompt="Summarize the given text. Return a concise summary and key points.",
            output_type=SummaryOutput,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> SummaryOutput:
        user_msg = state.get_latest_message("user")
        result = await self.call_model(user_msg, state)
        return result.output


# --- Translation Agent ---

class TranslationOutput(BaseModel):
    translated_text: str = Field(description="The translated text")
    source_language: str = Field(description="Detected source language")
    target_language: str = Field(description="Target language")


class TranslationAgent(BaseAgent[GlobalState, TranslationOutput]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="translator",
            system_prompt=(
                "Translate the given text to the requested language. "
                "If no target language is specified, translate to English."
            ),
            output_type=TranslationOutput,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> TranslationOutput:
        user_msg = state.get_latest_message("user")
        result = await self.call_model(user_msg, state)
        return result.output


print("Agents defined.")

## 3. Wrap agents as tools

Use `as_tool()` to convert each agent into a pydantic-ai `Tool`. The orchestrator's LLM will see these tools with their descriptions and decide when to call them.

Under the hood, `as_tool()` does this for each invocation:
```python
state = GlobalState()           # fresh state — no history
state.add_message("user", query)
output = await agent.run(state) # calls _run_implementation → call_model
return output.model_dump_json() # structured output as JSON string
```

In [ ]:
from orqest.agents import as_tool

# Create the specialized agents
summarizer = SummaryAgent(model=config.llm_model, api_key=config.llm_api_key)
translator = TranslationAgent(model=config.llm_model, api_key=config.llm_api_key)

# Wrap them as tools — one line each
summary_tool = as_tool(
    summarizer,
    description="Summarize a piece of text. Returns a concise summary and key points.",
)

translation_tool = as_tool(
    translator,
    description="Translate text to a target language. Specify the target language in your query.",
)

print(f"Tools created: {summary_tool.name}, {translation_tool.name}")

Tools created: summarizer, translator


## 4. Create the orchestrator

The orchestrator is a plain pydantic-ai `Agent` with the wrapped tools. It decides which tool to call based on the user's request.

In [ ]:
from pydantic_ai import Agent
from orqest.utils.llm_model import resolve_model

orchestrator = Agent(
    model=resolve_model(config.llm_model, api_key=config.llm_api_key),
    system_prompt=(
        "You are a helpful assistant with access to specialized tools. "
        "Use the summarizer tool when the user wants text summarized. "
        "Use the translator tool when the user wants text translated. "
        "After calling a tool, present the results clearly to the user."
    ),
    tools=[summary_tool, translation_tool],
)

print("Orchestrator ready.")

Orchestrator ready.


## 5. Run it

The orchestrator decides which specialized agent to invoke based on the user's request.

In [ ]:
# Ask for a summary — the orchestrator should call the summarizer tool
result = await orchestrator.run(
    "Please summarize this: Quantum computing uses qubits that can exist in "
    "superposition, representing both 0 and 1 simultaneously. Combined with "
    "entanglement, this enables solving certain problems exponentially faster "
    "than classical computers. Key applications include cryptography, drug "
    "discovery, and optimization."
)

print("Orchestrator response:")
print(result.output)

Orchestrator response:
Summary:
Quantum computing leverages qubits in superposition and entanglement to solve specific problems much faster than classical computers, with important applications in cryptography, drug discovery, and optimization.

Key Points:

- Qubits can represent 0 and 1 at the same time through superposition.
- Entanglement allows for complex computations beyond classical capabilities.
- Quantum computers can solve certain problems exponentially faster than classical computers.
- Major applications include cryptography, drug discovery, and optimization.


In [ ]:
# Ask for a translation — the orchestrator should call the translator tool
result = await orchestrator.run(
    "Translate this to French: The weather is beautiful today and I would like to go for a walk."
)

print("Orchestrator response:")
print(result.output)

Orchestrator response:
Here is the translation in French:
Il fait beau aujourd'hui et j'aimerais aller me promener.


## What's happening under the hood

1. The orchestrator's LLM receives the user message and sees two available tools: `summarizer` and `translator`
2. Based on the request, the LLM decides which tool to call and constructs the `query` parameter
3. `as_tool()` creates a **fresh `GlobalState`**, adds the query as a user message, and calls `agent.run()`
4. Inside `_run_implementation()`, `self.call_model()` passes the (empty) `state.message_history` to pydantic-ai
5. The structured output is serialized as JSON and returned to the orchestrator's LLM
6. The orchestrator formats the result for the user

Each tool call is independent — fresh state, no history. The specialized agents don't know about each other or the conversation.

### Stateful vs. stateless — same agents, different patterns

| | Example 01 (multi-turn) | Example 02 (agent-as-tool) |
|---|---|---|
| **State** | Shared `GlobalState` across turns | Fresh `GlobalState` per call |
| **History** | Accumulates via `call_model()` | Always empty |
| **Use case** | Conversational agent | Focused, one-shot specialist |

The same agent class works in both patterns — the invocation context (shared state vs. `as_tool()`) determines whether history accumulates.